In [ ]:
# ── Install all required packages ─────────────────────────────────────────────
!pip install xgboost lightgbm catboost pytorch-tabnet torch scikit-learn pandas numpy matplotlib seaborn

# Classification Models — Drug Dataset
---
**Dataset:** `drug_dataset_20k.csv` (20k rows, 10 cols)  
**Target:** `Drug` (5 classes: DrugA · DrugB · DrugC · DrugX · DrugY)  
**Features:** Age · Sex · BP · Cholesterol · Na_to_K · BMI · Smoker · Exercise_Level · Treatment_Duration_Days  
**Models:**
- **Baseline:** Logistic Regression · Naive Bayes · KNN · Decision Tree
- **Ensemble:** Random Forest · SVM · Gradient Boosting
- **Boosting:** XGBoost · LightGBM · CatBoost
- **Deep Learning:** TabNet · NODE · FT-Transformer · TabKAN  

**Metrics:** Accuracy · F1 (weighted) · Precision (weighted) · Recall (weighted) · ROC-AUC (macro-OvR)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('All imports successful.')

In [ ]:
# ── Load & Preprocess ─────────────────────────────────────────────────────────
df = pd.read_csv('../../../drug_dataset_20k.csv')
print(f'Shape: {df.shape}')
print(f'\nDrug distribution:\n{df["Drug"].value_counts()}')
print(f'\nMissing values:\n{df.isnull().sum()}')

# ── Encode categorical features ───────────────────────────────────────────────
le = {}
for col in ['Sex', 'BP', 'Cholesterol', 'Smoker', 'Exercise_Level']:
    le[col] = LabelEncoder()
    df[col + '_enc'] = le[col].fit_transform(df[col])

le_drug = LabelEncoder()
df['Drug_enc'] = le_drug.fit_transform(df['Drug'])

FEATURES = ['Age', 'Sex_enc', 'BP_enc', 'Cholesterol_enc', 'Na_to_K',
            'BMI', 'Smoker_enc', 'Exercise_Level_enc', 'Treatment_Duration_Days']
TARGET      = 'Drug_enc'
CLASSES     = le_drug.classes_
NUM_CLASSES = len(CLASSES)
N_FEATURES  = len(FEATURES)

# ── Sample 15 000 rows for speed ──────────────────────────────────────────────
df_s = df.sample(n=15000, random_state=SEED).reset_index(drop=True)
X    = df_s[FEATURES]
y    = df_s[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Scaled versions for distance/linear/neural models
scaler    = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURES, index=X_test.index)

# ── CatBoost — keep original string columns for native encoding ───────────────
CAT_COLS   = ['Sex', 'BP', 'Cholesterol', 'Smoker', 'Exercise_Level']
NUM_COLS   = ['Age', 'Na_to_K', 'BMI', 'Treatment_Duration_Days']
CB_COLS    = NUM_COLS + CAT_COLS
cat_idx    = [CB_COLS.index(c) for c in CAT_COLS]

df_cb      = df.sample(n=15000, random_state=SEED).reset_index(drop=True)
X_cb       = df_cb[CB_COLS]
y_cb       = df_cb[TARGET]
X_cb_tr, X_cb_te, y_cb_tr, y_cb_te = train_test_split(
    X_cb, y_cb, test_size=0.20, random_state=SEED, stratify=y_cb
)

print(f'\nTrain : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Classes ({NUM_CLASSES}): {CLASSES}')
print(f'Features ({N_FEATURES}): {FEATURES}')

In [ ]:
# ── Helper Functions ──────────────────────────────────────────────────────────
ALL_METRICS = {}

def compute_metrics(name, y_true, y_pred, y_prob=None):
    """Compute & store classification metrics."""
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='weighted')
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted')
    if y_prob is not None and np.ndim(y_prob) == 2:
        roc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    else:
        roc = np.nan
    metrics = {
        'Accuracy' : round(float(acc),  4),
        'F1'       : round(float(f1),   4),
        'Precision': round(float(prec), 4),
        'Recall'   : round(float(rec),  4),
        'ROC-AUC'  : round(float(roc),  4) if not np.isnan(roc) else np.nan
    }
    ALL_METRICS[name] = metrics
    print(f"\n{'═'*50}")
    print(f"  {name}")
    print(f"{'═'*50}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v}")
    return metrics

def plot_cm(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)
    ax.set_title(f'{title}\nConfusion Matrix', fontsize=11)
    ax.tick_params(axis='x', rotation=45)

def plot_class_f1(y_true, y_pred, title, ax):
    rpt = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)
    f1s = [rpt[c]['f1-score'] for c in CLASSES]
    ax.bar(CLASSES, f1s, color=sns.color_palette('viridis', NUM_CLASSES))
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Drug Class', fontsize=10)
    ax.set_ylabel('F1-Score', fontsize=10)
    ax.set_title(f'{title}\nPer-Class F1-Score', fontsize=11)
    ax.tick_params(axis='x', rotation=45)

# ── PyTorch generic training loop ─────────────────────────────────────────────
def train_pytorch(model, X_tr, y_tr, X_te, y_te,
                  epochs=60, batch_size=256, lr=1e-3):
    X_tr_t = torch.FloatTensor(np.array(X_tr))
    y_tr_t = torch.LongTensor(np.array(y_tr))
    X_te_t = torch.FloatTensor(np.array(X_te))
    loader  = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                         batch_size=batch_size, shuffle=True)
    opt     = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit    = nn.CrossEntropyLoss()
    history = {'loss': [], 'acc': []}
    for ep in range(epochs):
        model.train()
        ep_loss = 0.0
        for Xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(Xb), yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item() * len(yb)
        sched.step()
        history['loss'].append(ep_loss / len(y_tr_t))
        model.eval()
        with torch.no_grad():
            preds = model(X_te_t).argmax(dim=1).numpy()
        history['acc'].append(accuracy_score(y_te, preds))
        if (ep + 1) % 10 == 0:
            print(f'  Epoch {ep+1:3d}/{epochs}  loss={history["loss"][-1]:.4f}  acc={history["acc"][-1]:.4f}')
    return history

def pytorch_predict(model, X):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(np.array(X)))
        probs  = F.softmax(logits, dim=1).numpy()
    return probs.argmax(axis=1), probs

def plot_dl_history(history, title, ax):
    ax2 = ax.twinx()
    ax.plot(history['loss'], 'b-', lw=2, label='Train Loss')
    ax2.plot(history['acc'], 'r-', lw=2, label='Test Acc')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color='blue')
    ax2.set_ylabel('Accuracy', color='red')
    ax.set_title(f'{title}\nTraining History')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

print('Helper functions defined.')

---
## Part 1 — Baseline Models
Simple, fast, and interpretable — the first-try models.

---
## Model 1 — Logistic Regression
The simplest linear classifier for multiclass problems.  
Learns a linear boundary: **P(y=k|x) = softmax(Wₖx + bₖ)**

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
lr_model = LogisticRegression(max_iter=1000, C=1.0, multi_class='auto',
                               solver='lbfgs', random_state=SEED)
lr_model.fit(X_train_s, y_train)
y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)

m_lr = compute_metrics('Logistic Regression', y_test, y_pred_lr, y_prob_lr)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_lr, 'Logistic Regression', axes[0])
plot_class_f1(y_test, y_pred_lr, 'Logistic Regression', axes[1])

coef_mag = np.abs(lr_model.coef_).mean(axis=0)
axes[2].barh(FEATURES, coef_mag, color='steelblue')
axes[2].set_xlabel('Mean |Coefficient|')
axes[2].set_title('Logistic Regression\nMean Feature Coefficient Magnitude')
plt.tight_layout(); plt.show()

---
## Model 2 — Naive Bayes
Probabilistic model based on Bayes' theorem with feature-independence assumption.  
Fast and works well with limited data. Uses Gaussian likelihood for continuous features.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
nb = GaussianNB()
nb.fit(X_train_s, y_train)
y_pred_nb = nb.predict(X_test_s)
y_prob_nb = nb.predict_proba(X_test_s)

m_nb = compute_metrics('Naive Bayes', y_test, y_pred_nb, y_prob_nb)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_nb, 'Naive Bayes', axes[0])
plot_class_f1(y_test, y_pred_nb, 'Naive Bayes', axes[1])

# Learned class priors
axes[2].bar(CLASSES, nb.class_prior_, color=sns.color_palette('Set2', NUM_CLASSES))
axes[2].set_ylim(0, 1)
axes[2].set_xlabel('Drug Class'); axes[2].set_ylabel('Prior Probability')
axes[2].set_title('Naive Bayes\nLearned Class Priors')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

---
## Model 3 — K-Nearest Neighbors (KNN)
Classifies a point by majority vote of its **K nearest neighbors** in feature space.  
No training phase — prediction is distance-based at inference time.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=7, weights='distance', n_jobs=-1)
knn.fit(X_train_s, y_train)
y_pred_knn = knn.predict(X_test_s)
y_prob_knn = knn.predict_proba(X_test_s)

m_knn = compute_metrics('KNN', y_test, y_pred_knn, y_prob_knn)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_knn, 'KNN (K=7)', axes[0])
plot_class_f1(y_test, y_pred_knn, 'KNN (K=7)', axes[1])

# K vs Accuracy sweep
k_vals = range(1, 26)
k_acc  = [accuracy_score(y_test,
           KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
           .fit(X_train_s, y_train).predict(X_test_s))
          for k in k_vals]
axes[2].plot(k_vals, k_acc, 'bo-', ms=5, lw=1.5)
axes[2].axvline(7, color='red', linestyle='--', label='K=7 (used)')
axes[2].set_xlabel('K (Neighbors)'); axes[2].set_ylabel('Test Accuracy')
axes[2].set_title('KNN\nAccuracy vs K'); axes[2].legend()
plt.tight_layout(); plt.show()

---
## Model 4 — Decision Tree
A single flowchart model that recursively splits data on the best feature threshold.  
Fully interpretable — splits can be visualized and explained.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=SEED)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)

m_dt = compute_metrics('Decision Tree', y_test, y_pred_dt, y_prob_dt)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_dt, 'Decision Tree', axes[0])
plot_class_f1(y_test, y_pred_dt, 'Decision Tree', axes[1])

fi_dt = pd.Series(dt.feature_importances_, index=FEATURES).sort_values()
fi_dt.plot.barh(ax=axes[2], color='coral')
axes[2].set_xlabel('Importance')
axes[2].set_title('Decision Tree\nFeature Importances')
plt.tight_layout(); plt.show()

# Depth vs Accuracy (overfitting check)
depths = range(2, 16)
tr_acc = []; te_acc = []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, random_state=SEED)
    m.fit(X_train, y_train)
    tr_acc.append(accuracy_score(y_train, m.predict(X_train)))
    te_acc.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(depths, tr_acc, 'b-o', ms=6, label='Train Acc')
plt.plot(depths, te_acc, 'r-o', ms=6, label='Test Acc')
plt.axvline(8, color='grey', linestyle='--', label='depth=8 (used)')
plt.xlabel('Max Depth'); plt.ylabel('Accuracy')
plt.title('Decision Tree — Train vs Test Accuracy by Depth')
plt.legend(); plt.tight_layout(); plt.show()

---
## Part 2 — Classical Ensemble Models
Combine multiple models to reduce errors and improve stability.

---
## Model 5 — Random Forest
An ensemble of many decision trees trained on bootstrap samples with random feature subsets.  
Reduces variance via **bagging** — the forest votes on the final class.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=150, max_depth=None, min_samples_leaf=5,
                             random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)

m_rf = compute_metrics('Random Forest', y_test, y_pred_rf, y_prob_rf)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_rf, 'Random Forest', axes[0])
plot_class_f1(y_test, y_pred_rf, 'Random Forest', axes[1])

fi_rf = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fi_rf.plot.barh(ax=axes[2], color=sns.color_palette('magma', N_FEATURES))
axes[2].set_xlabel('Importance')
axes[2].set_title('Random Forest\nFeature Importances')
plt.tight_layout(); plt.show()

# n_estimators vs Accuracy
n_trees = [10, 25, 50, 75, 100, 125, 150]
rf_acc  = [accuracy_score(y_test,
           RandomForestClassifier(n_estimators=n, min_samples_leaf=5, random_state=SEED, n_jobs=-1)
           .fit(X_train, y_train).predict(X_test))
           for n in n_trees]
plt.figure(figsize=(8, 4))
plt.plot(n_trees, rf_acc, 'g-o', ms=7)
plt.xlabel('Number of Trees'); plt.ylabel('Test Accuracy')
plt.title('Random Forest — Test Accuracy vs Number of Trees')
plt.tight_layout(); plt.show()

---
## Model 6 — Support Vector Machine (SVM)
Finds the **maximum-margin hyperplane** separating classes in high-dimensional space.  
Uses `LinearSVC` with calibration for scalability; regularization is controlled by `C`.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
base_svc = LinearSVC(C=1.0, max_iter=3000, random_state=SEED)
svm      = CalibratedClassifierCV(base_svc, cv=3)
svm.fit(X_train_s, y_train)
y_pred_svm = svm.predict(X_test_s)
y_prob_svm = svm.predict_proba(X_test_s)

m_svm = compute_metrics('SVM (Linear)', y_test, y_pred_svm, y_prob_svm)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_svm, 'SVM (Linear)', axes[0])
plot_class_f1(y_test, y_pred_svm, 'SVM (Linear)', axes[1])

# C vs Accuracy sweep
C_vals  = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
svm_acc = [accuracy_score(y_test,
           CalibratedClassifierCV(LinearSVC(C=c, max_iter=3000, random_state=SEED), cv=3)
           .fit(X_train_s, y_train).predict(X_test_s))
           for c in C_vals]
axes[2].semilogx(C_vals, svm_acc, 'mo-', ms=8)
axes[2].axvline(1.0, color='grey', linestyle='--', label='C=1.0 (used)')
axes[2].set_xlabel('C'); axes[2].set_ylabel('Test Accuracy')
axes[2].set_title('SVM\nAccuracy vs Regularization C'); axes[2].legend()
plt.tight_layout(); plt.show()

---
## Model 7 — Gradient Boosting (GBM)
Sequentially builds trees where each new tree **corrects the errors** of the previous ensemble.  
Minimizes a loss function via gradient descent in function space.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
gbm = GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.1,
                                  subsample=0.8, random_state=SEED)
gbm.fit(X_train, y_train)
y_pred_gbm = gbm.predict(X_test)
y_prob_gbm = gbm.predict_proba(X_test)

m_gbm = compute_metrics('GBM', y_test, y_pred_gbm, y_prob_gbm)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_gbm, 'GBM', axes[0])
plot_class_f1(y_test, y_pred_gbm, 'GBM', axes[1])

fi_gbm = pd.Series(gbm.feature_importances_, index=FEATURES).sort_values()
fi_gbm.plot.barh(ax=axes[2], color='teal')
axes[2].set_xlabel('Importance')
axes[2].set_title('GBM\nFeature Importances')
plt.tight_layout(); plt.show()

# Staged accuracy (training curve)
stage_acc = [accuracy_score(y_test, y_s)
             for y_s in gbm.staged_predict(X_test)]
plt.figure(figsize=(8, 4))
plt.plot(stage_acc, 'b-', lw=1.5)
plt.xlabel('Boosting Round'); plt.ylabel('Test Accuracy')
plt.title('GBM — Test Accuracy per Boosting Round')
plt.tight_layout(); plt.show()

---
## Part 3 — High-Performance Boosting
The "gold standard" for tabular data competitions.

---
## Model 8 — XGBoost
Highly efficient, scalable, regularized gradient boosting with second-order gradients.  
Adds L1/L2 regularization and column subsampling for better generalization.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=SEED, n_jobs=-1
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)], verbose=False)
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)

m_xgb = compute_metrics('XGBoost', y_test, y_pred_xgb, y_prob_xgb)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_xgb, 'XGBoost', axes[0])
plot_class_f1(y_test, y_pred_xgb, 'XGBoost', axes[1])

fi_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values()
fi_xgb.plot.barh(ax=axes[2], color='darkorange')
axes[2].set_xlabel('Importance (gain)')
axes[2].set_title('XGBoost\nFeature Importances')
plt.tight_layout(); plt.show()

# Training loss curve
results = xgb_model.evals_result()
plt.figure(figsize=(8, 4))
plt.plot(results['validation_0']['mlogloss'], 'r-', lw=2, label='Test logloss')
plt.xlabel('Boosting Round'); plt.ylabel('Log Loss')
plt.title('XGBoost — Test Log Loss per Round')
plt.legend(); plt.tight_layout(); plt.show()

---
## Model 9 — LightGBM
Optimized for speed and large datasets using **leaf-wise** (best-first) tree growth.  
Uses histogram-based splits and GOSS sampling for much faster training than GBM/XGBoost.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
lgbm_model = lgb.LGBMClassifier(
    n_estimators=200, num_leaves=31, max_depth=-1,
    learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm_model.fit(X_train, y_train,
               eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(20, verbose=False),
                          lgb.log_evaluation(period=-1)])
y_pred_lgbm = lgbm_model.predict(X_test)
y_prob_lgbm = lgbm_model.predict_proba(X_test)

m_lgbm = compute_metrics('LightGBM', y_test, y_pred_lgbm, y_prob_lgbm)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_lgbm, 'LightGBM', axes[0])
plot_class_f1(y_test, y_pred_lgbm, 'LightGBM', axes[1])

fi_lgbm = pd.Series(lgbm_model.feature_importances_, index=FEATURES).sort_values()
fi_lgbm.plot.barh(ax=axes[2], color='seagreen')
axes[2].set_xlabel('Importance (split count)')
axes[2].set_title('LightGBM\nFeature Importances')
plt.tight_layout(); plt.show()

---
## Model 10 — CatBoost
Best at handling **categorical features natively** — no manual encoding needed.  
Uses ordered boosting and target statistics for robust categorical embeddings.

In [ ]:
# ── Fit — pass original string categoricals directly ─────────────────────────
cb_model = cb.CatBoostClassifier(
    iterations=200, depth=6, learning_rate=0.1,
    cat_features=cat_idx,
    random_seed=SEED, verbose=0
)
cb_model.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_te, y_cb_te))
y_pred_cb = cb_model.predict(X_cb_te).flatten()
y_prob_cb = cb_model.predict_proba(X_cb_te)

m_cb = compute_metrics('CatBoost', y_cb_te, y_pred_cb, y_prob_cb)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_cb_te, y_pred_cb, 'CatBoost', axes[0])
plot_class_f1(y_cb_te, y_pred_cb, 'CatBoost', axes[1])

fi_cb = pd.Series(cb_model.get_feature_importance(), index=CB_COLS).sort_values()
colors_cb = ['coral' if c in CAT_COLS else 'steelblue' for c in fi_cb.index]
fi_cb.plot.barh(ax=axes[2], color=colors_cb)
axes[2].set_xlabel('Importance')
axes[2].set_title('CatBoost\nFeature Importances (orange=categorical)')
import matplotlib.patches as mpatches
axes[2].legend(handles=[
    mpatches.Patch(color='coral',    label='Categorical'),
    mpatches.Patch(color='steelblue', label='Numeric')
], fontsize=9)
plt.tight_layout(); plt.show()

---
## Part 4 — Modern Deep Learning for Tabular Data
Neural architectures that use attention and splines to rival tree ensembles.

---
## Model 11 — TabNet
Uses **sequential attention** to select which features to focus on at each step,  
mimicking the decision-tree structure while remaining end-to-end differentiable.

In [ ]:
# ── Fit ───────────────────────────────────────────────────────────────────────
tabnet = TabNetClassifier(
    n_d=32, n_a=32, n_steps=3, gamma=1.3,
    n_independent=2, n_shared=2,
    cat_idxs=[], cat_dims=[], cat_emb_dim=1,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={'step_size': 10, 'gamma': 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    seed=SEED, verbose=10
)
tabnet.fit(
    X_train_s.values, y_train.values,
    eval_set=[(X_test_s.values, y_test.values)],
    max_epochs=60, patience=15,
    batch_size=512, virtual_batch_size=128
)
y_pred_tn = tabnet.predict(X_test_s.values)
y_prob_tn = tabnet.predict_proba(X_test_s.values)

m_tn = compute_metrics('TabNet', y_test, y_pred_tn, y_prob_tn)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_tn, 'TabNet', axes[0])
plot_class_f1(y_test, y_pred_tn, 'TabNet', axes[1])

fi_tn = pd.Series(tabnet.feature_importances_, index=FEATURES).sort_values()
fi_tn.plot.barh(ax=axes[2], color='purple')
axes[2].set_xlabel('Attention Importance')
axes[2].set_title('TabNet\nFeature Importances (Attention Masks)')
plt.tight_layout(); plt.show()

---
## Model 12 — NODE (Neural Oblivious Decision Ensembles)
Trains differentiable **oblivious decision trees** (same split at each depth level) inside a neural network.  
Each tree uses soft routing via sigmoid gates; gradients flow through the entire tree structure.

In [ ]:
# ── NODE Architecture ─────────────────────────────────────────────────────────
class ODST(nn.Module):
    """Oblivious Decision Soft Tree — one layer of NODE."""
    def __init__(self, in_features, num_trees, depth, output_dim):
        super().__init__()
        self.depth    = depth
        self.num_trees = num_trees
        n_splits = num_trees * depth
        self.feature_proj = nn.Linear(in_features, n_splits, bias=False)
        self.thresholds   = nn.Parameter(torch.zeros(n_splits))
        self.leaf_scores  = nn.Parameter(
            torch.randn(num_trees, 2**depth, output_dim) * 0.01
        )
        nn.init.normal_(self.feature_proj.weight, std=0.01)

    def forward(self, x):
        B = x.shape[0]
        vals = (self.feature_proj(x) - self.thresholds).view(B, self.num_trees, self.depth)
        go_right = torch.sigmoid(10.0 * vals)  # hard-ish split
        go_left  = 1.0 - go_right
        # Build leaf probabilities: (B, num_trees, 2^depth)
        leaf_prob = torch.ones(B, self.num_trees, 1, device=x.device)
        for d in range(self.depth):
            l = leaf_prob * go_left[:, :, d:d+1]
            r = leaf_prob * go_right[:, :, d:d+1]
            leaf_prob = torch.cat([l, r], dim=2)
        # (B,T,2^D) x (T,2^D,out) → (B, out)
        return torch.einsum('btn,tno->bo', leaf_prob, self.leaf_scores) / self.num_trees


class NODEClassifier(nn.Module):
    """Ensemble of ODST layers — full NODE model."""
    def __init__(self, in_features, num_classes, num_trees=128, depth=4, num_layers=2):
        super().__init__()
        self.bn_in = nn.BatchNorm1d(in_features)
        self.trees = nn.ModuleList([
            ODST(in_features, num_trees, depth, num_classes)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        x = self.bn_in(x)
        return sum(tree(x) for tree in self.trees)


print('NODE architecture defined.')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
node_model = NODEClassifier(N_FEATURES, NUM_CLASSES, num_trees=128, depth=4, num_layers=2)
print(f'NODE params: {sum(p.numel() for p in node_model.parameters()):,}')

hist_node = train_pytorch(node_model, X_train_s, y_train, X_test_s, y_test,
                          epochs=80, batch_size=512, lr=1e-3)

y_pred_node, y_prob_node = pytorch_predict(node_model, X_test_s)
m_node = compute_metrics('NODE', y_test, y_pred_node, y_prob_node)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_node, 'NODE', axes[0])
plot_class_f1(y_test, y_pred_node, 'NODE', axes[1])
plot_dl_history(hist_node, 'NODE', axes[2])
plt.tight_layout(); plt.show()

---
## Model 13 — FT-Transformer (Feature Tokenizer + Transformer)
Applies the **Transformer architecture** to tabular data.  
Each feature is embedded into a token; a CLS token aggregates cross-feature attention for the final prediction.

In [ ]:
# ── FT-Transformer Architecture ───────────────────────────────────────────────
class FTTransformer(nn.Module):
    """Feature Tokenizer + Transformer for tabular classification."""
    def __init__(self, in_features, num_classes, d_token=64,
                 n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        # Per-feature linear tokenizer (one linear layer per feature)
        self.feat_embed = nn.ModuleList([
            nn.Linear(1, d_token) for _ in range(in_features)
        ])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=4 * d_token,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, num_classes)
        )

    def forward(self, x):
        B = x.shape[0]
        # Tokenize each feature: list → (B, n_features, d_token)
        tokens = torch.stack(
            [emb(x[:, i:i+1]) for i, emb in enumerate(self.feat_embed)], dim=1
        )
        # Prepend CLS token
        cls    = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)   # (B, 1+n_feat, d_token)
        tokens = self.transformer(tokens)
        return self.head(tokens[:, 0])             # CLS output → logits


print('FT-Transformer architecture defined.')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
ftt_model = FTTransformer(N_FEATURES, NUM_CLASSES, d_token=64, n_heads=4, n_layers=3)
print(f'FT-Transformer params: {sum(p.numel() for p in ftt_model.parameters()):,}')

hist_ftt = train_pytorch(ftt_model, X_train_s, y_train, X_test_s, y_test,
                         epochs=80, batch_size=512, lr=1e-3)

y_pred_ftt, y_prob_ftt = pytorch_predict(ftt_model, X_test_s)
m_ftt = compute_metrics('FT-Transformer', y_test, y_pred_ftt, y_prob_ftt)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_ftt, 'FT-Transformer', axes[0])
plot_class_f1(y_test, y_pred_ftt, 'FT-Transformer', axes[1])
plot_dl_history(hist_ftt, 'FT-Transformer', axes[2])
plt.tight_layout(); plt.show()

---
## Model 14 — TabKAN (Kolmogorov-Arnold Networks for Tabular Data)
Replaces fixed activation functions with **learnable spline functions** (B-spline / RBF basis).  
Every edge in the network has its own trainable 1D function, enabling higher expressiveness.

In [ ]:
# ── TabKAN Architecture ───────────────────────────────────────────────────────
class KANLayer(nn.Module):
    """KAN linear layer: learnable spline activations via Gaussian RBF basis."""
    def __init__(self, in_features, out_features, grid_size=5):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        # Residual base weight (SiLU activation)
        self.base_weight   = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        # Spline coefficients: (out, in, grid)
        self.spline_weight = nn.Parameter(
            torch.randn(out_features, in_features, grid_size) * 0.01
        )
        self.scale = nn.Parameter(torch.ones(out_features, in_features))
        # Fixed grid over [-2, 2]
        self.register_buffer('grid', torch.linspace(-2, 2, grid_size))
        self.sigma = (4.0 / (grid_size - 1)) * 0.5  # RBF bandwidth

    def forward(self, x):
        # Base branch: SiLU(x) @ W_base^T
        base = F.silu(x) @ self.base_weight.T
        # Spline branch: Gaussian RBF basis
        rbf   = torch.exp(-0.5 * ((x.unsqueeze(-1) - self.grid) / self.sigma) ** 2)
        # rbf: (B, in, grid)  spline_weight * scale: (out, in, grid)
        spline = torch.einsum(
            'big,oig->bo', rbf, self.spline_weight * self.scale.unsqueeze(-1)
        )
        return base + spline


class TabKANClassifier(nn.Module):
    """Full KAN network for tabular classification."""
    def __init__(self, in_features, num_classes,
                 hidden_sizes=(64, 32), grid_size=5):
        super().__init__()
        self.bn_in = nn.BatchNorm1d(in_features)
        dims   = [in_features] + list(hidden_sizes) + [num_classes]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(KANLayer(dims[i], dims[i+1], grid_size))
            if i < len(dims) - 2:
                layers.append(nn.BatchNorm1d(dims[i+1]))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(self.bn_in(x))


print('TabKAN architecture defined.')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
tabkan = TabKANClassifier(N_FEATURES, NUM_CLASSES,
                          hidden_sizes=(64, 32), grid_size=5)
print(f'TabKAN params: {sum(p.numel() for p in tabkan.parameters()):,}')

hist_kan = train_pytorch(tabkan, X_train_s, y_train, X_test_s, y_test,
                         epochs=80, batch_size=512, lr=1e-3)

y_pred_kan, y_prob_kan = pytorch_predict(tabkan, X_test_s)
m_kan = compute_metrics('TabKAN', y_test, y_pred_kan, y_prob_kan)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plot_cm(y_test, y_pred_kan, 'TabKAN', axes[0])
plot_class_f1(y_test, y_pred_kan, 'TabKAN', axes[1])
plot_dl_history(hist_kan, 'TabKAN', axes[2])
plt.tight_layout(); plt.show()

---
## Model Comparison — All 14 Classifiers

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
metrics_df = pd.DataFrame(ALL_METRICS).T
metrics_df = metrics_df.sort_values('Accuracy', ascending=False)

print('\n' + '═'*72)
print('  MODEL COMPARISON — Ranked by Accuracy (higher is better)')
print('═'*72)

display(metrics_df.style
    .background_gradient(subset=['Accuracy', 'F1', 'Precision', 'Recall'], cmap='Greens')
    .background_gradient(subset=['ROC-AUC'], cmap='Blues')
    .format(lambda v: f'{v:.4f}' if not (isinstance(v, float) and np.isnan(v)) else 'N/A')
)

In [ ]:
# ── Bar-chart comparison for every metric ─────────────────────────────────────
metrics_to_plot = ['Accuracy', 'F1', 'Precision', 'Recall', 'ROC-AUC']
n_models = len(metrics_df)
palette  = sns.color_palette('tab20', n_models)

fig, axes = plt.subplots(2, 3, figsize=(22, 11))
axes = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    sdf = metrics_df[[metric]].dropna().sort_values(metric, ascending=True)
    bars = axes[i].barh(sdf.index, sdf[metric],
                        color=palette[:len(sdf)], edgecolor='white')
    for bar, val in zip(bars, sdf[metric]):
        axes[i].text(bar.get_width() * 0.98, bar.get_y() + bar.get_height()/2,
                     f'{val:.4f}', va='center', ha='right', fontsize=8.5,
                     color='white', fontweight='bold')
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_xlim(0, 1.05)
    axes[i].tick_params(axis='y', labelsize=8)

axes[5].axis('off')
fig.suptitle('Classification Models — Metric Comparison\n(Drug Dataset · 14 Models)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── F1 vs Accuracy scatter — model positioning ────────────────────────────────
plt.figure(figsize=(11, 7))
for (model, row), color in zip(metrics_df.iterrows(), palette):
    plt.scatter(row['Accuracy'], row['F1'], s=200, color=color,
                zorder=3, edgecolors='white', linewidths=0.7)
    plt.annotate(model, (row['Accuracy'], row['F1']),
                 textcoords='offset points', xytext=(6, 3), fontsize=8.5)

# Diagonal reference (F1 ≈ Accuracy for balanced classes)
mn = metrics_df[['Accuracy', 'F1']].min().min() - 0.02
mx = metrics_df[['Accuracy', 'F1']].max().max() + 0.02
plt.plot([mn, mx], [mn, mx], 'k--', lw=1, alpha=0.4, label='F1 = Accuracy')

plt.xlabel('Accuracy  (higher is better →)', fontsize=11)
plt.ylabel('F1-Score (weighted)  (higher is better ↑)', fontsize=11)
plt.title('Model Positioning — Accuracy vs F1\n(best models: top-right)', fontsize=12)
plt.legend(fontsize=9); plt.tight_layout(); plt.show()